# Synthetic EHR — Full Pipeline (TimeDiff + Multinomial Diffusion)

Run cells **top to bottom**. Everything saves directly to Google Drive.

| Stage | Cells | Time |
|-------|-------|------|
| Setup | 1–3 | 2 min |
| Preprocessing | 4–7 | 20–60 min per dataset |
| Training | 8–10 | 4–8 hr per dataset (L4 GPU) |
| Generation | 11 | 5–10 min per dataset |
| Evaluation | 12–13 | 20–30 min |

> **Tip:** Run Cells 8, 9, 10 in **3 separate Colab Pro sessions** to train all datasets in parallel.
> If disconnected: remount Drive (Cell 1) and re-run — training auto-resumes from last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, torch
PROJECT = '/content/drive/MyDrive/synthetic-ehr-ddpm'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

for d in ['data/processed', 'data/synthetic', 'results/plots']:
    os.makedirs(d, exist_ok=True)

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print('GPU:', device_name)
if 'NO GPU' in device_name:
    print('  Go to Runtime > Change runtime type > GPU (L4 recommended)')
print('Working dir:', os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'numpy', 'pandas', 'scikit-learn', 'scipy',
    'matplotlib', 'seaborn', 'wandb', 'pyyaml', 'tqdm', 'einops'])
print('Done.')

In [ ]:
import wandb
wandb.login()  # paste API key from https://wandb.ai/authorize

---
## Stage 1 — Preprocessing

Each cell preprocesses one dataset and saves:
- `{ds}_train/val/test.npy` — normalized features  
- `{ds}_train/val/test_mask.npy` — binary missingness masks (**new**)  
- `{ds}_scaler.pkl`

Raw data must already be in `data/raw/mimic3/`, `data/raw/mimic4/`, `data/raw/eicu/` on Drive.

In [ ]:
import importlib, sys
for key in [k for k in sys.modules if 'loaders' in k or 'data' in k]:
    del sys.modules[key]
import data.loaders.mimic3 as _m
importlib.reload(_m)
_m.main(raw_dir='data/raw/mimic3', out_dir='data/processed')
print('MIMIC-III done.')

In [ ]:
import importlib, sys
for key in [k for k in sys.modules if 'loaders' in k or 'data' in k]:
    del sys.modules[key]
import data.loaders.mimic4 as _m
importlib.reload(_m)
_m.main(raw_dir='data/raw/mimic4', out_dir='data/processed')
print('MIMIC-IV done.')

In [ ]:
import importlib, sys
for key in [k for k in sys.modules if 'loaders' in k or 'data' in k]:
    del sys.modules[key]
import data.loaders.eicu as _m
importlib.reload(_m)
_m.main(raw_dir='data/raw/eicu', out_dir='data/processed')
print('eICU done.')

In [ ]:
import numpy as np, os

all_ok = True
for ds in ['mimic3', 'mimic4', 'eicu']:
    print(f'\n{ds.upper()}:')
    for split in ['train', 'val', 'test']:
        p  = f'data/processed/{ds}_{split}.npy'
        mp = f'data/processed/{ds}_{split}_mask.npy'
        if os.path.exists(p):
            d = np.load(p)
            print(f'  {split:5s} data : {d.shape}  min={d.min():.3f} max={d.max():.3f}')
        else:
            print(f'  MISSING: {p}'); all_ok = False
        if os.path.exists(mp):
            m = np.load(mp)
            print(f'  {split:5s} mask : {m.shape}  obs_rate={m.mean():.2%}')
        else:
            print(f'  MISSING mask: {mp}'); all_ok = False

print('\nAll files ready!' if all_ok else '\nSome files missing — re-run preprocessing cells.')

---
## Stage 2 — Training

- Trains for **10,000 epochs**, checkpoints every 500 epochs to Drive
- **Auto-resumes** from last checkpoint if session disconnects
- Run Cells 8, 9, 10 in **3 separate Colab Pro tabs** to train in parallel

> Expected loss at convergence: **0.009 – 0.013**

In [ ]:
# Train TimeDiff — MIMIC3  (~4-8 hr on L4)
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'train_timediff.py',
     '--config', 'configs/timediff_mimic3.yaml',
     '--run_name', 'timediff_mimic3'],
    cwd='/content/drive/MyDrive/synthetic-ehr-ddpm',
    stdout=None, stderr=None  # stream output live to notebook
)
print('Exit code:', result.returncode)

In [ ]:
# Train TimeDiff — MIMIC4  (~4-8 hr on L4)
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'train_timediff.py',
     '--config', 'configs/timediff_mimic4.yaml',
     '--run_name', 'timediff_mimic4'],
    cwd='/content/drive/MyDrive/synthetic-ehr-ddpm',
    stdout=None, stderr=None  # stream output live to notebook
)
print('Exit code:', result.returncode)

In [ ]:
# Train TimeDiff — EICU  (~4-8 hr on L4)
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'train_timediff.py',
     '--config', 'configs/timediff_eicu.yaml',
     '--run_name', 'timediff_eicu'],
    cwd='/content/drive/MyDrive/synthetic-ehr-ddpm',
    stdout=None, stderr=None  # stream output live to notebook
)
print('Exit code:', result.returncode)

In [ ]:
import os, torch

all_ok = True
for ds in ['mimic3', 'mimic4', 'eicu']:
    for ckpt_name in ['last.pt', 'best.pt']:
        p = f'results/timediff_{ds}/{ckpt_name}'
        if os.path.exists(p):
            size_mb = os.path.getsize(p) / 1e6
            ckpt = torch.load(p, map_location='cpu', weights_only=False)
            epoch = ckpt.get('epoch', '?')
            print(f'  OK  {p}  ({size_mb:.1f} MB)  epoch={epoch}')
        else:
            print(f'  MISSING: {p}'); all_ok = False

print('\nAll checkpoints ready.' if all_ok else '\nSome missing — finish training first.')

---
## Stage 3 — Generation

Generates 10,000 synthetic patients per dataset.
Multinomial mask diffusion runs automatically inside `diffusion.sample()` — no extra args needed.

Output: `data/synthetic/timediff_{ds}.npy`

In [ ]:
import os, subprocess, sys, numpy as np
os.makedirs('data/synthetic', exist_ok=True)

for ds in ['mimic3', 'mimic4', 'eicu']:
    ckpt = f'results/timediff_{ds}/best.pt'
    out  = f'data/synthetic/timediff_{ds}.npy'
    if not os.path.exists(ckpt):
        print(f'SKIP {ds} — checkpoint not found'); continue
    print(f'Generating {ds}...')
    result = subprocess.run(
        [sys.executable, 'generate.py',
         '--model',      'timediff',
         '--config',     f'configs/timediff_{ds}.yaml',
         '--checkpoint', ckpt,
         '--n_samples',  '10000',
         '--output',     out,
         '--data_dir',   'data/processed'],
        cwd='/content/drive/MyDrive/synthetic-ehr-ddpm'
    )
    if result.returncode == 0 and os.path.exists(out):
        s = np.load(out)
        print(f'  Saved {s.shape}  min={s.min():.4f}  max={s.max():.4f}  mean={s.mean():.4f}')
    else:
        print(f'  ERROR — exit code {result.returncode}')

print('Generation complete.')

---
## Stage 4 — Evaluation

Runs 5 metrics per dataset: Discriminative Score, Predictive MAE, MMD, NNAA, Visualisations.
Results saved to `results/metrics.csv` on Drive.

In [ ]:
import os, subprocess, sys

metrics_path = 'results/metrics.csv'
if os.path.exists(metrics_path):
    os.remove(metrics_path)
    print('Cleared old metrics.csv')

for ds in ['mimic3', 'mimic4', 'eicu']:
    synthetic = f'data/synthetic/timediff_{ds}.npy'
    if not os.path.exists(synthetic):
        print(f'SKIP {ds} — synthetic file not found'); continue
    os.makedirs(f'results/plots/{ds}/timediff', exist_ok=True)
    print(f'Evaluating {ds}...')
    result = subprocess.run(
        [sys.executable, 'evaluate.py',
         '--dataset',   ds,
         '--model',     'timediff',
         '--synthetic', synthetic,
         '--data_dir',  'data/processed',
         '--device',    'cuda'],
        cwd='/content/drive/MyDrive/synthetic-ehr-ddpm'
    )
    if result.returncode != 0:
        print(f'  ERROR evaluating {ds}')

print('Evaluation complete.')

In [ ]:
import pandas as pd, os

metrics_path = 'results/metrics.csv'
if not os.path.exists(metrics_path):
    print('No results yet — run Cell 12 first.')
else:
    df = pd.read_csv(metrics_path)
    df['Disc_Paper'] = (df['Disc_Score'].astype(float) - 0.5).abs().round(4)

    print('=' * 80)
    print('FINAL RESULTS')
    print('=' * 80)
    print(df[['Dataset', 'Model', 'Disc_Score', 'Disc_Paper', 'Pred_MAE', 'MMD', 'NNAA']].to_string(index=False))
    print('=' * 80)
    print('Disc_Paper = |Disc_Score - 0.5|  matches paper Table 1 format (lower=better)')
    print('NNAA       = AA_test privacy score (0.5 = most private, matches paper Table 2)')

    print()
    print('--- Paper TimeDiff reference ---')
    ref = pd.DataFrame({
        'Dataset':    ['mimic3', 'mimic4', 'eicu'],
        'Disc_Paper': [0.028,    0.030,    0.015],
        'AA_test':    [0.574,    0.517,    0.537],
    })
    print(ref.to_string(index=False))

---
## Appendix — Utilities

In [ ]:
# Check training progress (run anytime while training is running)
import os, torch

for ds in ['mimic3', 'mimic4', 'eicu']:
    p = f'results/timediff_{ds}/last.pt'
    if os.path.exists(p):
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        epoch = ckpt.get('epoch', '?')
        total = ckpt.get('cfg', {}).get('epochs', 10000)
        pct = f"{(epoch + 1) / total * 100:.1f}%" if isinstance(epoch, int) else '?'
        print(f'{ds:8s}  epoch {epoch + 1 if isinstance(epoch, int) else epoch}/{total}  ({pct})')
    else:
        print(f'{ds:8s}  no checkpoint yet')

In [ ]:
# List all output files with sizes
import os

for d in ['data/processed', 'data/synthetic', 'results']:
    if not os.path.exists(d): continue
    print(f'\n{d}/')
    for root, _, files in os.walk(d):
        for f in sorted(files):
            full = os.path.join(root, f)
            rel  = os.path.relpath(full, d)
            size = os.path.getsize(full)
            unit, val = ('MB', size/1e6) if size > 1e6 else ('KB', size/1e3)
            print(f'  {rel:55s} {val:7.1f} {unit}')

In [ ]:
# Re-generate + re-evaluate a single dataset
import os, subprocess, sys

DS = 'mimic3'  # change to mimic4 or eicu as needed

ckpt = f'results/timediff_{DS}/best.pt'
out  = f'data/synthetic/timediff_{DS}.npy'

print(f'Generating {DS}...')
subprocess.run(
    [sys.executable, 'generate.py',
     '--model', 'timediff', '--config', f'configs/timediff_{DS}.yaml',
     '--checkpoint', ckpt, '--n_samples', '10000',
     '--output', out, '--data_dir', 'data/processed'],
    cwd='/content/drive/MyDrive/synthetic-ehr-ddpm'
)

print(f'Evaluating {DS}...')
subprocess.run(
    [sys.executable, 'evaluate.py',
     '--dataset', DS, '--model', 'timediff',
     '--synthetic', out, '--data_dir', 'data/processed', '--device', 'cuda'],
    cwd='/content/drive/MyDrive/synthetic-ehr-ddpm'
)